# Quanto Options pricing under CARMA framework using Monte-Carlo method

## Model parameters definition

In [26]:
#--------------------------------------------------------------------- Imports

import numpy as np
import pandas as pd
from scipy.linalg import expm
import matplotlib.pyplot as plt
from scipy.stats import norm
from datetime import datetime
from dateutil.relativedelta import relativedelta
from scipy.integrate import quad


#------------------------------------------------------------------- Parameters

# CARMA Parameters
sig_X = 0.036050456023551204
b0_X  = 1.0
b1_X  = -1.69107499948807
b2_X  = 0.0856060699524467
a1_X  = 2.7417368516678
a2_X  = 1.85827342924239
a3_X  = 0.161163594914844

sig_Y = 0.76963298150931
b0_Y  = 1.0
b1_Y  = 0.584420888567256
a1_Y  = 0.834602554797278
a2_Y  = 0.02473622057743

# NIG Parameters
mu_X    = 0.032544431425439106
delta_X = 0.3350081939529935
alpha_X = 0.9821566545385684
beta_X  = -0.09493955549441858

# Gaussian parameters
mu_Y    = 0.004626935939183486
sigma_Y = 0.9619333921285353

# Coupling parameter
gamma = 0.0017485173776097822


#-------------------------------------------------------- Seasonal trend parameters for Y
'''From Gabriel's calibration'''

inputs_Y = {
    "intercept": 10.771397,
    "trend":      0.000006,

    "day_cos_1": -2.773628,
    "day_sin_1": -1.343241,
    "day_cos_2":  0.531266,
    "day_sin_2":  0.000657,
    "day_cos_3":  0.138549,
    "day_sin_3":  0.080022,

    "year_cos_1": -8.946115,
    "year_sin_1": -3.027131,
}

inter_Y = {
    (0,0):  1.607345, (0,1): -0.312687,
    (1,0):  0.756860, (1,1): -0.159530,
    (2,0):  0.222841, (2,1): -0.054528,
    (3,0):  0.260909, (3,1):  0.050530,
    (4,0): -0.384971, (4,1):  0.065902,
    (5,0): -0.039503, (5,1): -0.000543,
}

P_DAY = 24
P_YEAR = 365.25 * 24


#---------------------------------------------------------- Matrices for log prices X
'''The YUIMA convention holds'''

# Companion matrix A_X of shape (3, 3)
AX = np.array([[0,         1,          0],
               [0,         0,          1],
               [-a3_X,    -a2_X,   -a1_X]])

# Noise vector B_X of shape (3, 1)
BX =  np.array([[0   ], [0   ], [sig_X ]])   

# Selection vector c_X of shape (3, 1)
cX = np.array([b0_X, b1_X, b2_X])                          # put b2_X for CARMA(3,2)

# Coupling parameter Γ of shape (3, 1)
Gamma = np.array([[0.0],
                  [0.0],
                  [gamma]])

#---------------------------------------------------------- Matrices for Temperature Y
'''The YUIMA convention holds'''

# Companion matrix A_Y of shape (2, 2)
AY = np.array([[    0,        1  ], 
               [-a2_Y,      -a1_Y]])

# Noise vector B_X of shape (2, 1)
BY = np.array([[ 0 ], [sig_Y]])   

# Selection vector c_X of shape (2, 1)
cY = np.array([b0_Y, b1_Y])     


### Functions definition

In [27]:
#---------------------------------------------------- Precomputation of structural matrices

def structural_matrices(params):
    AX, BX, Gamma = params['AX'], params['BX'], params['Gamma']
    AY, BY = params['AY'], params['BY']
    T, n_steps, dt = params['T'], params['n_steps'], params['dt']
    

    eAXdt = expm(AX * dt) 
    eAYdt = expm(AY * dt) 

    MXX = np.linalg.inv(AX) @ (eAXdt - np.eye(len(AX))) @ BX
    MYY = np.linalg.inv(AY) @ (eAYdt - np.eye(len(AY))) @ BY
    MYX = np.linalg.inv(AX) @ (eAXdt - np.eye(len(AX))) @ Gamma
    
    return {'dt'      : dt,
            'eAXdt'   : eAXdt, # (3, 3)
            'eAYdt'   : eAYdt, # (2, 2)
            'MXX' : MXX,       # (3, 1)
            'MYX' : MYX,       # (3, 1)
            'MYY' : MYY}       # (2, 1)

    
#----------------------------------------------- Deterministic function for log prices seasonnal part

file = r"C:\Users\nolan\PyCharmMiscProject\seasonalities.csv"
df = pd.read_csv(file, index_col=0, parse_dates=True)

seasonal_series = df["log_price_seasonal"].copy()
seasonal_series.index = pd.to_datetime(seasonal_series.index, utc=True)

lookup = (pd.DataFrame({"values"    : seasonal_series.values,            # return a dictionnary of tuples (m, dow, h)
                        "month"  : seasonal_series.index.month,       
                        "dow"    : seasonal_series.index.dayofweek,      # 0=monday ... 6=sunday
                        "hour"   : seasonal_series.index.hour})
          .groupby(["month", "dow", "hour"])["values"]                    # for example, (3, 2, 12) stands for 12h on a wednesday in march
          .mean()                                                        # we take the mean of the series values on the same dayofweek, hour, month
          .to_dict())

def Lambda_S(params):                                            # consistent with Paraschiv's method
    t = params['end_date']
    key = (t.month, t.dayofweek, t.hour)
    return lookup.get(key, np.nan)
    
#----------------------------------------------- Deterministic function for temperature seasonnal part

def day_features(t):
    return np.array([
        np.cos(2*np.pi*1*t/P_DAY), np.sin(2*np.pi*1*t/P_DAY),
        np.cos(2*np.pi*2*t/P_DAY), np.sin(2*np.pi*2*t/P_DAY),
        np.cos(2*np.pi*3*t/P_DAY), np.sin(2*np.pi*3*t/P_DAY),
    ])

def year_features(t):
    return np.array([
        np.cos(2*np.pi*1*t/P_YEAR),
        np.sin(2*np.pi*1*t/P_YEAR),
    ])

def Lambda_Y(params):
    t = params['O']
    y = inputs_Y["intercept"] + inputs_Y["trend"] * t
    y += inputs_Y["day_cos_1"] * np.cos(2*np.pi*1*t/P_DAY)
    y += inputs_Y["day_sin_1"] * np.sin(2*np.pi*1*t/P_DAY)
    y += inputs_Y["day_cos_2"] * np.cos(2*np.pi*2*t/P_DAY)
    y += inputs_Y["day_sin_2"] * np.sin(2*np.pi*2*t/P_DAY)
    y += inputs_Y["day_cos_3"] * np.cos(2*np.pi*3*t/P_DAY)
    y += inputs_Y["day_sin_3"] * np.sin(2*np.pi*3*t/P_DAY)
    y += inputs_Y["year_cos_1"] * np.cos(2*np.pi*1*t/P_YEAR)
    y += inputs_Y["year_sin_1"] * np.sin(2*np.pi*1*t/P_YEAR)
    d = day_features(t)
    v = year_features(t)
    for (i, j), c in inter_Y.items():
        y += c * d[i] * v[j]
    return y



#--------------------------------------------------------------- Discount factor


def discount_factor(r, t):
    r = r / 8760                                 # hourly interest rate for an annual of r percent
    return np.exp(-r*t)


#----------------------------------------------------------- Lévy increments with Gaussian distribution


def sample_gaussian(mu, sigma):
    T, n_steps, dt, N = params['T'], params['n_steps'], params['dt'], params['N']
    return np.random.normal(loc=mu, scale=sigma, size=(int(n_steps), int(N)))

    
#----------------------------------------------------------- Lévy increments with NIG distribution


def sample_inverse_gaussian(delta, gamma, size):
    M = np.random.randn(*size)                      # M ~ N(0,1)    
    V = M**2                                       # V ~ χ²(1) (chi square law with 1 degree of freedom)

    X_1 = (delta / gamma                           # First solution of the quadratic equation V = [(gamma . X - delta)^2]/X
          + V / (2 * gamma**2)
          - np.sqrt(delta * V / gamma**3 + (V / (2 * gamma**2))**2))                                  
           
    X_2 = (delta / gamma)**2 / X_1                 # Second solution
    
    U  = np.random.uniform(size=size)              # U ~ U[0,1]
    return np.where(U <= delta / (delta + gamma * X_1), X_1, X_2) # return X_1 if U <= mu_ig / (mu_ig + X_1), else return X_2

# X ~ NIG(alpha, beta, delta, mu) if X = mu + beta Y + sqrt(Y) Z 
# where Z ~ N(0,1)
# and Y ~ IG(delta, gamma), with gamma = sqrt[alpha^2 - beta^2]

    
def sample_NIG(alpha, beta, delta, mu):      
    T, n_steps, dt, N = params['T'], params['n_steps'], params['dt'], params['N'] 
    gamma = np.sqrt(alpha**2 - beta**2)
    Y = sample_inverse_gaussian(delta, gamma, size=(int(n_steps), int(N))) # put size = (n_steps, 1) for a path, or size = (n_steps, N) for all paths
    Z = np.random.randn(int(n_steps), int(N))                              # put size = (n_steps, 1) for a path, or size = (n_steps, N) for all paths
    return mu + beta * Y + np.sqrt(Y) * Z


#-------------------------------------------------------------- Coupled paths simulation

def simulate_paths(params, matrices):
    
    N, n_steps, T, end_date = params['N'], params['n_steps'], params['T'], params['end_date']
    dt = matrices['dt']
    alpha_X, beta_X, delta_X, mu_X = params['alpha_X'], params['beta_X'], params['delta_X']*dt, params['mu_X']*dt
    alpha_Y, beta_Y, delta_Y, mu_Y = params['alpha_Y'], params['beta_Y'], params['delta_Y']*dt, params['mu_Y']*dt

    np.random.seed(42)                                      # optionnal
    
    dLX = sample_NIG(alpha_X, beta_X, delta_X, mu_X)  # (n_steps, N)
    dLY = sample_NIG(alpha_Y, beta_Y, delta_Y, mu_Y)  # (n_steps, N)

    ZX = np.zeros((N, 3))   # (N, 3)
    ZY = np.zeros((N, 2))   # (N, 2)

    eAXdt = matrices['eAXdt'] # (3, 3)
    eAYdt = matrices['eAYdt'] # (2, 2)
    MXX   = matrices['MXX']   # (3, 1)
    MYX   = matrices['MYX']   # (3, 1)
    MYY   = matrices['MYY']   # (2, 1)

    for t in range(int(n_steps)):          # for each timestamp increment, we update Z_t^i to Z_{t+dt}^i, for all i in {1,...,N}
        dLX_t = dLX[t]   # (N,)            # we select the t-th ligne (numpy squeeze the dimension, the result is an array)
        dLY_t = dLY[t]   # (N,)

        ZX = (ZX @ eAXdt.T                       #(N, 3)
              + np.outer(dLX_t, MXX.flatten())   #(N, 3)     
              + np.outer(dLY_t, MYX.flatten()))

        ZY = (ZY @ eAYdt.T                       #(N, 2)
              + np.outer(dLY_t, MYY.flatten()))  #(N, 2)

    '''
    np.outer function :
    
    a = np.array([1, 2, 3])       
    b = np.array([4, 5, 6])
    result = np.outer(a, b)
    print(result)
    -->[[ 4 5 6]
        [ 8 10 12]
        [12 15 18]]
    shape(result) = (len(a), len(b))
    '''

    cX, cY = params['cX'], params['cY']

    X_T = ZX @ cX                              # (N,) terminal values for deseasonnalised shifted log prices
    Y_T = ZY @ cY                              # (N,) terminal values for deseasonnalised shifted temperature

    S_T     = np.exp(Lambda_S(params) + X_T) - 1000       # (N,) terminal values for spot prices
    Y_hat_T = Lambda_Y(params) + Y_T                # (N,) terminal values for temperature

    return S_T, Y_hat_T



#------------------------------------------------------------------- Monte-Carlo pricer


def Monte_Carlo_price(payoff, params, matrices, confidence=0.95):
    
    S_T, Y_hat_T = simulate_paths(params, matrices)
    K_S, K_Y = params['K_S'], params['K_Y']
    payoff_distribution = payoff(S_T, K_S, Y_hat_T, K_Y)               # (N,)

    price = discount_factor(params['r'], params['T']) * np.mean(payoff_distribution)
    standard_error  = discount_factor(params['r'], params['T']) * np.std(payoff_distribution) / np.sqrt(params['N'])

    z  = norm.ppf((1 + confidence) / 2)
    confidence_interval = (price - z * standard_error, price + z * standard_error)

    return price, confidence_interval, standard_error



#------------------------------------------------------------- Payoff functions


''' Spot forward'''
def spot_forward(S_T, K_S, Y_hat_T, K_Y):
    return S_T


''' Plain energy call'''
def payoff_plain_energy_call(S_T, K_S, Y_hat_T, K_Y):
    return np.maximum(S_T - K_S, 0)


''' Plain energy put'''
def payoff_plain_energy_put(S_T, K_S, Y_hat_T, K_Y):
    return np.maximum(K_S - S_T, 0)

    
''' Plain temperature call'''
def payoff_plain_temperature_call(S_T, K_S, Y_hat_T, K_Y):
    return np.maximum(Y_hat_T - K_Y, 0)

        
''' Plain temperature put'''
def payoff_plain_temperature_put(S_T, K_S, Y_hat_T, K_Y):
    return np.maximum(K_Y - Y_hat_T, 0)

''' Barrier temperature'''
def payoff_temperature_barrier(S_T, K_S, Y_hat_T, K_Y):
    return (Y_hat_T > K_Y).astype(float)
    

''' Quanto call'''
def payoff_quanto_call(S_T, K_S, Y_hat_T, K_Y):
    return np.maximum(S_T - K_S, 0) * (Y_hat_T > K_Y).astype(float)





## Plain energy call pricing

#### $\mathbb{E}[(S_T - K_S)^+]$

#### Monte-Carlo price computation

In [28]:
#------------------------------------------------------------ Parameters dictionnary

start_date = pd.Timestamp("2026-05-13 12:00:00")
end_date = pd.Timestamp("2026-06-12 19:00:00")
origin_date = pd.Timestamp("2026-01-01 00:00:00")      # Date corresponding to t=0 in Lambda_Y(t)

T = (end_date - start_date).total_seconds() / 3600     # Distance from maturity (hour)
O = (end_date - origin_date).total_seconds() / 3600    # Distance from origin (hour)


params = {'AX': AX, 'BX': BX, 'cX': cX, 'Gamma': Gamma,
          'AY': AY, 'BY': BY, 'cY': cY,
          'mu_X': mu_X, 'delta_X': delta_X, 'alpha_X': alpha_X, 'beta_X': beta_X,
          'mu_Y': mu_Y, 'delta_Y': delta_Y, 'alpha_Y': alpha_Y, 'beta_Y': beta_Y,
          
          'start_date': start_date,
          'end_date': end_date,
          'origin_date' : origin_date,   
          'T' : T,                      
          'O' : O,                      
          
          'N' : 10_000,
          'dt' : 1,
          'n_steps': T,                            # because dt is fixed to 1
          'r' : 0,
          'K_S' : 130,
          'K_Y' : 10}

#--------------------------------------------------- Structural matrices dictionnary

matrices = structural_matrices(params)

#-------------------------------------------------------- Price computation

MC_price, confidence_interval, standard_error = Monte_Carlo_price(payoff_plain_energy_call, params, matrices)

print(f"Monte-Carlo price  : {MC_price:.4f}")
print(f"Std error     : {standard_error:.4f}")
print(f"Confidence interval 95%         : [{confidence_interval[0]:.4f}, {confidence_interval[1]:.4f}]")
print(f"Relative error : {standard_error/MC_price*100:.2f}%")

Monte-Carlo price  : 21.2131
Std error     : 0.2941
Confidence interval 95%         : [20.6366, 21.7896]
Relative error : 1.39%


In [12]:
print(np.exp(Lambda_S(params))-1000)

136.02465319417365


### Call Put parity

##### The following relation should be satisfied : $C - P = \mathbb{E}[S_T] - K_S$

In [12]:
call, _, _ = Monte_Carlo_price(payoff_plain_energy_call, params, matrices)
put,  _, _ = Monte_Carlo_price(payoff_plain_energy_put,  params, matrices)
forward,   _, _ = Monte_Carlo_price(spot_forward, params, matrices)         #E[S_T]

print(f"C - P          : {call - put:.4f}")
print(f"E[S_T] - K_S   : {forward - params['K_S']:.4f}")

C - P          : 6.5748
E[S_T] - K_S   : 6.5748


## Plain temperature call pricing

#### $\mathbb{E}[(Y_T - K_Y)^+]$

In [25]:
#------------------------------------------------------------ Parameters dictionnary

start_date = pd.Timestamp("2026-05-13 12:00:00")
end_date = pd.Timestamp("2026-06-12 19:00:00")
origin_date = pd.Timestamp("2026-01-01 00:00:00")      # Date corresponding to t=0 in Lambda_Y(t)

T = (end_date - start_date).total_seconds() / 3600     # Distance from maturity (hour)
O = (end_date - origin_date).total_seconds() / 3600    # Distance from origin (hour)


params = {'AX': AX, 'BX': BX, 'cX': cX, 'Gamma': Gamma,
          'AY': AY, 'BY': BY, 'cY': cY,
          'mu_X': mu_X, 'delta_X': delta_X, 'alpha_X': alpha_X, 'beta_X': beta_X,
          'mu_Y': mu_Y, 'delta_Y': delta_Y, 'alpha_Y': alpha_Y, 'beta_Y': beta_Y,
          
          'start_date': start_date,
          'end_date': end_date,
          'origin_date' : origin_date,   
          'T' : T,                      
          'O' : O,                      
          
          'N' : 10_000,
          'dt' : 1,
          'n_steps': T,                            # because dt is fixed to 1
          'r' : 0,
          'K_S' : 130,
          'K_Y' : 15}

#--------------------------------------------------- Structural matrices dictionnary

matrices = structural_matrices(params)

#-------------------------------------------------------- Price computation

MC_price, confidence_interval, standard_error = Monte_Carlo_price(payoff_plain_temperature_call, params, matrices)

print(f"Monte-Carlo price  : {MC_price:.4f}")
print(f"Std error     : {standard_error:.4f}")
print(f"Confidence interval 95%         : [{confidence_interval[0]:.4f}, {confidence_interval[1]:.4f}]")
print(f"Relative error : {standard_error/MC_price*100:.2f}%")

Monte-Carlo price  : 4.5040
Std error     : 0.0418
Confidence interval 95%         : [4.4221, 4.5859]
Relative error : 0.93%


In [8]:
print(Lambda_Y(params))

18.811604367585648


## Barrier option on temperature

### $\mathbb{E}[\mathbb{1}_{Y_T > K_Y}]$

In [20]:

#------------------------------------------------------------ Parameters dictionnary

start_date = pd.Timestamp("2026-05-13 12:00:00")
end_date = pd.Timestamp("2026-06-12 19:00:00")
origin_date = pd.Timestamp("2026-01-01 00:00:00")      # Date corresponding to t=0 in Lambda_Y(t)

T = (end_date - start_date).total_seconds() / 3600     # Distance from maturity (hour)
O = (end_date - origin_date).total_seconds() / 3600    # Distance from origin (hour)


params = {'AX': AX, 'BX': BX, 'cX': cX, 'Gamma': Gamma,
          'AY': AY, 'BY': BY, 'cY': cY,
          'mu_X': mu_X, 'delta_X': delta_X, 'alpha_X': alpha_X, 'beta_X': beta_X,
          'mu_Y': mu_Y, 'delta_Y': delta_Y, 'alpha_Y': alpha_Y, 'beta_Y': beta_Y,
          
          'start_date': start_date,
          'end_date': end_date,
          'origin_date' : origin_date,   
          'T' : T,                      
          'O' : O,                      
          
          'N' : 10_000,
          'dt' : 1,
          'n_steps': T,                            # because dt is fixed to 1
          'r' : 0,
          'K_S' : 130,
          'K_Y' : 18}

#--------------------------------------------------- Structural matrices dictionnary

matrices = structural_matrices(params)

#-------------------------------------------------------- Price computation

MC_price, confidence_interval, standard_error = Monte_Carlo_price(payoff_temperature_barrier, params, matrices)

print(f"Monte-Carlo price  : {MC_price:.4f}")
print(f"Std error     : {standard_error:.4f}")
print(f"Confidence interval 95%         : [{confidence_interval[0]:.4f}, {confidence_interval[1]:.4f}]")
print(f"Relative error : {standard_error/MC_price*100:.2f}%")

Monte-Carlo price  : 0.5594
Std error     : 0.0050
Confidence interval 95%         : [0.5497, 0.5691]
Relative error : 0.89%


## Quanto option

### $\mathbb{E}[(Y_T - K_Y)^+\mathbb{1}_{Y_T > K_Y}]$

In [24]:

#------------------------------------------------------------ Parameters dictionnary

start_date = pd.Timestamp("2026-05-13 12:00:00")
end_date = pd.Timestamp("2026-06-12 19:00:00")
origin_date = pd.Timestamp("2026-01-01 00:00:00")      # Date corresponding to t=0 in Lambda_Y(t)

T = (end_date - start_date).total_seconds() / 3600     # Distance from maturity (hour)
O = (end_date - origin_date).total_seconds() / 3600    # Distance from origin (hour)


params = {'AX': AX, 'BX': BX, 'cX': cX, 'Gamma': Gamma,
          'AY': AY, 'BY': BY, 'cY': cY,
          'mu_X': mu_X, 'delta_X': delta_X, 'alpha_X': alpha_X, 'beta_X': beta_X,
          'mu_Y': mu_Y, 'delta_Y': delta_Y, 'alpha_Y': alpha_Y, 'beta_Y': beta_Y,
          
          'start_date': start_date,
          'end_date': end_date,
          'origin_date' : origin_date,   
          'T' : T,                      
          'O' : O,                      
          
          'N' : 10_000,
          'dt' : 1,
          'n_steps': T,                            # because dt is fixed to 1
          'r' : 0,
          'K_S' : 136,
          'K_Y' : 18}

#--------------------------------------------------- Structural matrices dictionnary

matrices = structural_matrices(params)

#-------------------------------------------------------- Price computation

MC_price, confidence_interval, standard_error = Monte_Carlo_price(payoff_quanto_call, params, matrices)

print(f"Monte-Carlo price  : {MC_price:.4f}")
print(f"Std error     : {standard_error:.4f}")
print(f"Confidence interval 95%         : [{confidence_interval[0]:.4f}, {confidence_interval[1]:.4f}]")
print(f"Relative error : {standard_error/MC_price*100:.2f}%")

Monte-Carlo price  : 14.2390
Std error     : 0.2635
Confidence interval 95%         : [13.7226, 14.7555]
Relative error : 1.85%
